In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
import sys
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src import io, flux_integrate, synthetic
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from scipy.ndimage import map_coordinates
import cv2

In [ ]:
def _sample_background(background, x, y):
    """
    Sample the background at coordinates (x, y) ∈ [0, 1].
 
    Parameters
    ----------
    background : float or (H, W) array-like
        Scalar constant, or a 2-D image defined on the unit square.
        The image is sampled with bilinear interpolation; x maps to columns,
        y maps to rows (origin at bottom-left, i.e. y=0 → last row).
    x, y : ndarray
        Query coordinates in [0, 1].
 
    Returns
    -------
    ndarray or float
    """
    if np.isscalar(background) or np.asarray(background).ndim == 0:
        return float(background)
 
    bg = np.asarray(background, dtype=float)
    if bg.ndim != 2:
        raise ValueError("`background` image must be 2-D (H, W).")
 
    H, W = bg.shape
    # Map unit coords → pixel coords.
    # x ∈ [0,1] → col ∈ [0, W-1],  y ∈ [0,1] → row ∈ [H-1, 0]  (flip so y=0 is bottom)
    cols = np.asarray(x, dtype=float) * (W - 1)
    rows = np.asarray(y, dtype=float) * (H - 1)
    return map_coordinates(bg, [rows.ravel(), cols.ravel()], order=1, mode="nearest").reshape(x.shape)
 
 
def make_2gaussian(
    # --- Ball 1 ---
    amp1: float = 1.0,
    pos1: tuple[float, float] = (0.2, 0.2),
    vel1: tuple[float, float] = (0.5, 0.3),
    sigma1: float = 0.08,
    # --- Ball 2 ---
    amp2: float = 0.8,
    pos2: tuple[float, float] = (0.8, 0.7),
    vel2: tuple[float, float] = (-0.4, 0.2),
    sigma2: float = 0.12,
    # --- Global ---
    background=0.0,
    wrap: bool = False,
):
    """
    Create a flux(x, y, t) function representing two moving Gaussian blobs
    on the unit spatiotemporal domain.
 
    Parameters
    ----------
    amp1, amp2 : float
        Peak amplitude of each Gaussian ball.
    pos1, pos2 : (float, float)
        (x0, y0) starting position of each ball at t=0, in [0, 1].
    vel1, vel2 : (float, float)
        (vx, vy) velocity in domain-lengths per unit time.
    sigma1, sigma2 : float
        Spatial standard deviation (isotropic) of each Gaussian in domain units.
    background : float or (H, W) array-like
        Constant baseline, or a 2-D image sampled over the unit square.
        The image is bilinearly interpolated; x → columns, y → rows (y=0 at bottom).
    wrap : bool
        Toroidal (periodic) boundary conditions when True.
    """
 
    def _gaussian(x, y, cx, cy, sigma):
        if wrap:
            dx = (x - cx + 0.5) % 1.0 - 0.5
            dy = (y - cy + 0.5) % 1.0 - 0.5
        else:
            dx = x - cx
            dy = y - cy
        return np.exp(-(dx**2 + dy**2) / (2.0 * sigma**2))
 
    def flux(x, y, t):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        t = np.asarray(t, dtype=float)
 
        cx1 = pos1[0] + vel1[0] * t
        cy1 = pos1[1] + vel1[1] * t
        cx2 = pos2[0] + vel2[0] * t
        cy2 = pos2[1] + vel2[1] * t
 
        g1 = amp1 * _gaussian(x, y, cx1, cy1, sigma1)
        g2 = amp2 * _gaussian(x, y, cx2, cy2, sigma2)
 
        return _sample_background(background, x, y) + g1 + g2
 
    flux.config = dict(
        amp1=amp1, pos1=pos1, vel1=vel1, sigma1=sigma1,
        amp2=amp2, pos2=pos2, vel2=vel2, sigma2=sigma2,
        background=background, wrap=wrap,
    )
    return flux
 
 
def make_2ball(
    # --- Ball 1 ---
    amp1: float = 1.0,
    pos1: tuple[float, float] = (0.2, 0.2),
    vel1: tuple[float, float] = (0.5, 0.3),
    radius1: float = 0.08,
    # --- Ball 2 ---
    amp2: float = 0.8,
    pos2: tuple[float, float] = (0.8, 0.7),
    vel2: tuple[float, float] = (-0.4, 0.2),
    radius2: float = 0.12,
    # --- Global ---
    background=0.0,
    wrap: bool = False,
):
    """
    Create a flux(x, y, t) function representing two moving hard-edge discs
    on the unit spatiotemporal domain.
 
    Parameters
    ----------
    amp1, amp2 : float
        Amplitude inside each disc.
    pos1, pos2 : (float, float)
        (x0, y0) centre at t=0, in [0, 1].
    vel1, vel2 : (float, float)
        (vx, vy) velocity in domain-lengths per unit time.
    radius1, radius2 : float
        Radius of each disc in domain units.
    background : float or (H, W) array-like
        Constant baseline, or a 2-D image sampled over the unit square.
        The image is bilinearly interpolated; x → columns, y → rows (y=0 at bottom).
    wrap : bool
        Toroidal (periodic) boundary conditions when True.
    """
 
    def _dist2(x, y, cx, cy):
        if wrap:
            dx = (x - cx + 0.5) % 1.0 - 0.5
            dy = (y - cy + 0.5) % 1.0 - 0.5
        else:
            dx = x - cx
            dy = y - cy
        return dx**2 + dy**2
 
    def flux(x, y, t):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        t = np.asarray(t, dtype=float)
 
        cx1 = pos1[0] + vel1[0] * t
        cy1 = pos1[1] + vel1[1] * t
        cx2 = pos2[0] + vel2[0] * t
        cy2 = pos2[1] + vel2[1] * t
 
        inside1 = _dist2(x, y, cx1, cy1) <= radius1**2
        inside2 = _dist2(x, y, cx2, cy2) <= radius2**2
 
        return _sample_background(background, x, y) + amp1 * inside1 + amp2 * inside2
 
    flux.config = dict(
        amp1=amp1, pos1=pos1, vel1=vel1, radius1=radius1,
        amp2=amp2, pos2=pos2, vel2=vel2, radius2=radius2,
        background=background, wrap=wrap,
    )
    return flux

In [ ]:
img = cv2.imread("../data/nyc-pic.webp")
img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # convert to grayscale
img = img[500:564, 500:564] / 255

In [ ]:
plt.imshow(img)

In [ ]:
vel1 = (0.7, 0.4)
vel2 = (0.4, -0.8)
# flux = make_2gaussian(
#     # --- Ball 1 ---
#     amp1=1.0,
#     pos1=(0.2, 0.2),
#     vel1=vel1,
#     sigma1=0.1,
#     # --- Ball 2 ---
#     amp2=1.0,
#     pos2=(0.8, 0.7),
#     vel2=vel2,
#     sigma2=0.0,
#     # --- Global ---
#     background=0.0,
#     wrap=False,
# )
flux = make_2ball(
    # --- Ball 1 ---
    amp1=1.0,
    pos1=(0.2, 0.2),
    vel1=vel1,
    radius1=0.1,
    # --- Ball 2 ---
    amp2=1.0,
    pos2=(0.4, 0.7),
    vel2=vel2,
    radius2=0.06,
    # --- Global ---
    background=img,
    wrap=False,
)

Nx = 64
Ny = 64
Nt = 100
xx = np.linspace(0, 1, Nx)
yy = np.linspace(0, 1, Ny)
tt = np.linspace(0, 1, Nt)
TT, YY, XX = np.meshgrid(tt, yy, xx, indexing='ij')
fluxgrid = flux(XX, YY, TT)
# fluxgrid_int = flux_integrate.integrate_voxel_grid(
#     flux,
#     W=1.0, H=1.0, tau=1.0,
#     grid_W=Nx, grid_H=Ny, grid_T=Nt,
#     method="midpoint",
#     n_samples=8,
# )
dataname = "synthetic_2ball"

In [ ]:
# fluxgrid, _ = io.load_video("../data/highway-trimmed.mp4", grayscale=True)
# H_src, W_src = fluxgrid.shape[1], fluxgrid.shape[2]
# fluxgrid, _ = io.load_video("../data/highway-trimmed.mp4", grayscale=True, resize_hw=(int(H_src * 0.2), int(W_src * 0.2)))
# Nt, Ny, Nx = fluxgrid.shape
# dataname = "highway"

fluxgrid, _ = io.load_video("../data/bullet-glass.mp4", grayscale=True)
H_src, W_src = fluxgrid.shape[1], fluxgrid.shape[2]
fluxgrid, _ = io.load_video("../data/bullet-glass.mp4", grayscale=True, resize_hw=(int(H_src * 0.05), int(W_src * 0.05)))
Nt, Ny, Nx = fluxgrid.shape
dataname = "bullet-glass"

In [ ]:
vel1norm = (vel1[0] / Nt * Nx, vel1[1] / Nt * Ny)
vel2norm = (vel2[0] / Nt * Nx, vel2[1] / Nt * Ny)
print("vel1norm:", vel1norm)
print("vel2norm:", vel2norm)

In [ ]:
1 / Nt * Nx

In [ ]:
ft = np.fft.fftfreq(Nt, d=1/Nt) * 1/Nt * Nt
fy = np.fft.fftfreq(Ny, d=1.0) * Ny
fx = np.fft.fftfreq(Nx, d=1.0) * Nx

In [ ]:
F = np.fft.fftn(fluxgrid, axes=(0, 1, 2))

In [ ]:
def normalize_velocities(velocities: np.ndarray, temporal_samples: int, spatial_samples: int):
    """
    Convert velocity bounds from pixel-per-frame units to a dimensionless normalized slope.

    This maps velocities defined on the discrete (x, t) grid to velocities in normalized
    coordinates x' = x / num_x and t' = t / num_t:

        v_norm = v_px_per_frame * (num_t / num_x)
               = (Δx/Δt) * (num_t/num_x)
               = (Δx/num_x) / (Δt/num_t)

    Parameters
    ----------
    velocities : np.ndarray
        Velocity bounds expressed in pixels per frame (Δx / Δt).
        Can be shape (2,) for [v_min, v_max] or any array of bounds.
    temporal_samples : int
        Number of frames (temporal samples) in the sequence.
    spatial_samples: int
        Number of pixels (spatial samples).

    Returns
    -------
    np.ndarray
        Velocity bounds in normalized units (dimensionless), i.e. slope in (x', t') space.
    """

    return velocities * (temporal_samples / spatial_samples)

def suppress_static_band(image, frequency, dc_bins=1, baseline_percentile=10.0):
    """
    Suppress near-DC components along the first axis and remove a column-wise baseline
    from a 2D frequency-frequency power plane.

    `image` is typically a projected power spectrum plane such as
    (ft x fx) or (ft x fy), obtained by summing |F(ft, fy, fx)|^2 over one spatial
    frequency dimension. Passing `frequency=ft` removes the temporal-DC band (ft ≈ 0),
    which corresponds to static / near-static content over time.

    Steps
    -----
    1) Zero out a narrow band around frequency ≈ 0 along the first axis:
         image[|frequency| < dc_bins * Δf, :] = 0
    2) Subtract a column-wise low-percentile baseline (estimated across the first axis),
       then clamp negatives to zero.

    Parameters
    ----------
    image : np.ndarray
        2D array of shape (len(frequency), M), e.g. a (ft x fx) or (ft x fy) power plane.
    frequency : np.ndarray
        1D array of bin centers corresponding to the first axis of `image`.
    dc_bins : int, optional
        Width of the suppressed near-DC band in units of frequency bins (default 1).
    baseline_percentile : float, optional
        Percentile used to estimate the per-column baseline across the first axis
        (default 10.0).

    Returns
    -------
    np.ndarray
        Processed 2D plane with near-DC rows removed and column-wise baseline suppressed.
    """

    image = image.copy()

    # Remove frequency ≈ 0
    delta_ft = abs(frequency[1] - frequency[0])
    dc_band = np.abs(frequency) < dc_bins * delta_ft
    image[dc_band, :] = 0

    # Column-wise low-percentile baseline subtraction
    baseline = np.percentile(image, baseline_percentile, axis=0, keepdims=True)
    image = image - baseline
    image[image < 0] = 0

    return image

def suppress_static_band_3d(F2: np.ndarray, ft: np.ndarray, dc_bins: int, baseline_percentile: float):
    """
    Minimal 3D version:
    - zero a band around ft=0 (or around center if ft is fftshifted)
    - subtract a baseline per-(fy,fx) column using a percentile across time
    """
    A = F2.copy()

    # zero near ft=0
    idx0 = int(np.argmin(np.abs(ft)))
    lo = max(0, idx0 - dc_bins)
    hi = min(A.shape[0], idx0 + dc_bins + 1)
    A[lo:hi, :, :] = 0

    # percentile baseline across time for each spatial freq bin (fy,fx)
    base = np.percentile(A, baseline_percentile, axis=0, keepdims=True)  # (1,Ny,Nx)
    A = A - base
    A[A < 0] = 0
    return A

def compute_energies_on_single_component_velocity_planes(
    velocities_px_frame: tuple[float, float, float],
    ft: np.ndarray,
    fy: np.ndarray,
    fx: np.ndarray,
    F: np.ndarray,
    num_t: int,
    num_x: int,
    direction: str = "x",
    dc_bins: int = 1,
    baseline_percentile: float = 10.0,
    sigma: float = 0.2,
):
    """
    Compute energy responses along single-component velocity planes in the 3D power spectrum.

    The 3D spectrum |F(ft, fy, fx)|² is projected onto either the (ft, fx) or (ft, fy) plane,
    static (ft ≈ 0) components are suppressed, and energy is accumulated along
    velocity-aligned lines for a set of candidate velocities.

    Parameters
    ----------
    velocities_px_frame : tuple of float
        (v_min, v_max, num_v) velocity range in pixels per frame.
    ft, fy, fx : np.ndarray
        Temporal and spatial frequency axes corresponding to F.
    F : np.ndarray
        3D Fourier transform with shape (len(ft), len(fy), len(fx)).
    num_t : int
        Number of temporal samples.
    num_x : int
        Number of spatial samples along the selected direction.
    direction : {"x", "y"}, optional
        Spatial component used for velocity evaluation (default "x").
    dc_bins : int, optional
        Width of the suppressed near-DC temporal band (default 1).
    baseline_percentile : float, optional
        Percentile for column-wise baseline subtraction (default 10.0).
    sigma : float, optional
        Width of the Gaussian mask orthogonal to the velocity line.

    Returns
    -------
    np.ndarray
        Energy values for each tested velocity.
    """
        
    # Velocity grid
    if len(velocities_px_frame) != 3:
        raise ValueError("velocities_px_frame must have length 3")
    v_bounds_px = np.linspace(*velocities_px_frame)
    v_bounds_norm = normalize_velocities(v_bounds_px, num_t, num_x)
    # Power spectrum
    F2 = np.abs(F) ** 2
    energies = []
    # Select projection plane
    if direction == "x":
        F2_plane = F2.sum(axis=1)
        F2_plane = suppress_static_band(F2_plane, ft, dc_bins, baseline_percentile)
        freq_axis = fx
    else:
        F2_plane = F2.sum(axis=2)
        F2_plane = suppress_static_band(F2_plane, ft, dc_bins, baseline_percentile)
        freq_axis = fy

    for v in v_bounds_norm:
        FT, FX = np.meshgrid(ft, freq_axis, indexing="ij")
        norm = np.sqrt(1 + v * v)
        d = (FT + v * FX) / norm
        # define a line of coefficients
        mask = np.exp(-(d**2) / (2 * sigma**2))
        hH, hW = mask.shape
        # Pad template to image size
        pad = np.zeros_like(F2_plane)
        pad[:hH, :hW] = mask

        energies.append(np.sum(F2_plane * pad))

    return np.array(energies)

def compute_energies_on_multiple_component_velocity_planes(
    velocities_px_frame,
    ft, fy, fx, F,
    num_t, num_x, num_y=None,
    dc_bins=1,
    baseline_percentile=10.0,
    sigma=0.2,
):
    if num_y is None:
        num_y = len(fy)

    # velocity grid
    v_px = np.linspace(*velocities_px_frame)
    vx = normalize_velocities(v_px, num_t, num_x)
    vy = normalize_velocities(v_px, num_t, num_y)
    Nv = len(vx)

    # power spectrum
    # F2 = np.abs(F)**2
    F2 = np.abs(F)
    F2s = suppress_static_band_3d(F2, ft, dc_bins, baseline_percentile)

    # flatten coordinates
    FT, FY, FX = np.meshgrid(ft, fy, fx, indexing="ij")
    P  = F2s.ravel()
    ft_f = FT.ravel()
    fx_f = FX.ravel()
    fy_f = FY.ravel()

    energies = np.zeros((Nv, Nv), dtype=np.float64)

    fy_f_bc = fy_f[:, None]
    vy_bc = vy[None, :]
    fvy_prod = fy_f_bc * vy_bc  # (N, Nv)

    # vectorized over vy
    for i, vxi in tqdm(enumerate(vx),total=len(vx)):
        base = ft_f + vxi * fx_f                 # (N,)
        xi_all = base[:, None] + fvy_prod  # (N, Nv)

        mask = np.abs(xi_all) < sigma            # boolean (N, Nv)
        energies[i, :] = (P[:, None] * mask).sum(axis=0)

    return energies

def compute_cfar_rs_2d(
    energies2d: np.ndarray,
    alpha: float,
    window: int = 5,
    guard: int = 1,
):
    """
    2D Rank-Sum (RS) Nonparametric CFAR.

    For each CUT at (r,c):
      - Neighborhood: rows [r-window, r+window], cols [c-window, c+window]
      - Guard(+CUT): rows [r-guard, r+guard], cols [c-guard, c+guard]
      - Training: neighborhood minus guard box
      - RS statistic: r_rc = count(E[r,c] >= training_vals)
      - Under H0, r_rc is uniform in {0,...,N_rc}
        => Pfa = (N_rc - T + 1)/(N_rc + 1)
        => threshold T derived from alpha

    Returns
    -------
    T : (H,W) float
        Rank statistic per CUT.
    E_bg : (H,W) float
        Background estimate per CUT (median of training cells).
    detections : (H,W) bool
        Detection mask.
    thresholds : (H,W) float
        Rank threshold per CUT.
    """

    E = np.asarray(energies2d, dtype=float)
    H, W = E.shape

    if window <= guard:
        raise ValueError("window must be > guard so training cells exist.")

    T = np.zeros((H, W), dtype=float)
    E_bg = np.zeros((H, W), dtype=float)
    thresholds = np.zeros((H, W), dtype=float)
    detections = np.zeros((H, W), dtype=bool)

    for r in range(H):
        r0 = max(0, r - window)
        r1 = min(H, r + window + 1)
        gr0 = max(0, r - guard)
        gr1 = min(H, r + guard + 1)

        for c in range(W):
            c0 = max(0, c - window)
            c1 = min(W, c + window + 1)
            gc0 = max(0, c - guard)
            gc1 = min(W, c + guard + 1)

            # neighborhood patch
            patch = E[r0:r1, c0:c1]

            # build training mask: True everywhere except guard box (including CUT)
            train_mask = np.ones_like(patch, dtype=bool)

            # guard box coords in patch coordinates
            pr0 = gr0 - r0
            pr1 = gr1 - r0
            pc0 = gc0 - c0
            pc1 = gc1 - c0
            train_mask[pr0:pr1, pc0:pc1] = False

            training_vals = patch[train_mask]
            N = training_vals.size

            if N == 0:
                # edge fallback (rare if window>guard and not tiny images)
                training_vals = E.ravel()
                N = training_vals.size

            # background statistic
            E_bg[r, c] = np.median(training_vals)

            # RS rank statistic
            r_rc = np.sum(E[r, c] >= training_vals)
            T[r, c] = r_rc

            # threshold from alpha (same as your 1D code)
            k = int(np.floor(alpha * (N + 1)))
            k = max(1, min(k, N + 1))
            T_thr = N - k + 1
            thresholds[r, c] = T_thr

            detections[r, c] = (r_rc >= T_thr)

    return T, E_bg, detections, thresholds

In [ ]:
import cv2
from pathlib import Path

def get_codec_for_format(format: str):
    """
    Get appropriate fourcc codec string for given video format.
    For MP4, tries avc1 first and falls back to mp4v if unavailable.
    """
    format = format.lower()
    if format == "mp4":
        return "avc1"
    elif format == "avi":
        return "FFV1"
    elif format == "mov":
        return "avc1"
    else:
        raise ValueError(f"I haven't added the codec for: {format}")

def to_video(
    frames: np.ndarray, path, res_scale=1.0, playback_fps=None, gamma=1.0, cmap=None, fileformat=None,
    vmin=None, vmax=None, quantile=None, framenames=None
):
    """
    Saves video frame arrays to a video file or sequence of PNGs. If path has no extension, 
    it is treated as a directory and individual image files are saved.

    Args:
        frames (np.ndarray): (T x H x W x C) (RGB) or (T x H x W) (intensity) video frames.
        path (str or Path): output video file path or directory for image files.
        res_scale (float): resolution scaling factor with nearest neighbor interpolation.
        cmap: ignored if frames are RGB; otherwise, matplotlib colormap name or object.
        fileformat (str or None): video format (e.g., "mp4", "avi"), or image format (e.g., "png");
            if None, inferred from path suffix.
        quantile (float or None): if not None, use quantiles to determine vmin and vmax for normalization
            (ignored if vmin or vmax are specified).
    """
    path = Path(path)
    if cmap is None:
        cmap = "viridis"
    cmap_fn = plt.get_cmap(cmap)
    is_rgb = False
    if frames.ndim == 4:
        if frames.shape[3] == 3:
            is_rgb = True
        else:
            raise ValueError("4D frames array must have shape (T, H, W, 3) for RGB video")
    elif frames.ndim == 3:
        is_rgb = False
    else:
        raise ValueError("frames must be a 3D or 4D numpy array")

    # compute a normalized intensity in [0,1] for colormap input
    if vmax is None:
        if quantile is not None:
            vmax = float(np.quantile(frames, quantile))
        else:
            vmax = float(np.max(frames))
    if vmin is None:
        if quantile is not None:
            vmin = float(np.quantile(frames, 1 - quantile))
        else:
            vmin = float(np.min(frames))
            if vmin >= 0:
                vmin = 0.0

    H, W = frames.shape[1], frames.shape[2]
    if res_scale != 1.0:
        out_W = int(W * res_scale)
        out_H = int(H * res_scale)
    else:
        out_W = W
        out_H = H
    # if path is a directory, write individual image files
    is_video_file = path.suffix in [".mp4", ".avi", ".mov", ".mkv"]
    if not is_video_file:
        path.mkdir(parents=True, exist_ok=True)
        if fileformat is None:
            fileformat = "png"
    else:
        if playback_fps is None:
            raise ValueError("playback_fps must be specified if saving a video file")
        path.parent.mkdir(parents=True, exist_ok=True)
        if fileformat is None:
            fileformat = path.suffix[1:].lower()
        codec = get_codec_for_format(fileformat)
        fourcc = cv2.VideoWriter_fourcc(*codec)
        vidwriter = cv2.VideoWriter(str(path), fourcc, playback_fps, (out_W, out_H), isColor=True)

    max_frames = len(frames)

    if not is_video_file:
        allpaths = []
    for i in tqdm(range(max_frames), desc="Writing video frames"):
        intensity = (np.clip(frames[i], vmin, vmax) - vmin) / (vmax - vmin)  # normalize to [0,1]
        if gamma != 1:
            intensity = intensity ** gamma
        if is_rgb:
            rgb_mapped = (intensity * 255.0).astype(np.uint8)  # (H,W,3) in RGB
        else:
            # apply matplotlib colormap -> returns RGBA in [0,1]
            rgba_mapped = cmap_fn(intensity)  # shape (H,W,4)
            rgb_mapped = (rgba_mapped[..., :3] * 255.0).astype(np.uint8)  # (H,W,3) in RGB
        bgr_mapped = rgb_mapped[..., ::-1]  # convert to BGR for OpenCV
        if res_scale != 1.0:
            bgr_mapped = cv2.resize(bgr_mapped, (out_W, out_H), interpolation=cv2.INTER_NEAREST)

        if is_video_file:
            vidwriter.write(bgr_mapped)
        else:
            if framenames is None:
                frame_path = path / f"frame_{i:05d}.{fileformat}"
            else:
                frame_path = path / f"{framenames[i]}.{fileformat}"
            if fileformat.lower() == "png":
                # higher compression level because there's thousands of frames
                # reminder for anyone reading here; IT'S LOSSLESS COMPRESSION BECAUSE IT'S A PNG
                cv2.imwrite(str(frame_path), bgr_mapped, [cv2.IMWRITE_PNG_COMPRESSION, 5])
            else:
                cv2.imwrite(str(frame_path), bgr_mapped)
            allpaths.append(frame_path)
    if is_video_file:
        vidwriter.release()
    if not is_video_file:
        return allpaths
    return path

In [ ]:
to_video(fluxgrid, f"../results/{dataname}_flux.mp4", playback_fps=30, cmap="gray")

In [ ]:
velrange = (-1.0, 1.0, 100)
velstoprobe = np.linspace(*velrange)
# energies = compute_energies_on_single_component_velocity_planes(
#     (-0.15, 0.15, 100),
#     ft, fy, fx, F,
#     10, 10
# )
energies = compute_energies_on_multiple_component_velocity_planes(
    velrange,
    ft, fy, fx, F,
    num_t=Nt, num_x=Nx, num_y=Ny,
    dc_bins=1,
    baseline_percentile=10.0,
    sigma=0.2,
)
_, E_bg, det, thr = compute_cfar_rs_2d(
    energies, alpha=1e-3, window=30, guard=1
)

In [ ]:
from src.plane_scoring_detection import detection

detector = detection.FourierMotionDetector.from_bounds(velrange, sigma=0.4)
result = detector.detect(fluxgrid)

In [ ]:
plt.imshow(result.detections)

In [ ]:
result.detected_velocities

In [ ]:
energies = result.energies
argmax_idx = np.unravel_index(np.argmax(energies), energies.shape)
velocities_grid = np.meshgrid(velstoprobe, velstoprobe, indexing="ij")
vx_max, vy_max = velocities_grid[0][argmax_idx], velocities_grid[1][argmax_idx]
plt.imshow(energies, extent=(velrange[0], velrange[1], velrange[0], velrange[1]), origin='lower')
det_idxs = np.argwhere(det)
plt.scatter(velstoprobe[det_idxs[:, 1]], velstoprobe[det_idxs[:, 0]], color='cyan', s=10, label='detections')
plt.colorbar()
plt.scatter(*vel1norm[::-1], color='red', label='vel1', marker="x")
plt.scatter(*vel2norm[::-1], color='blue', label='vel2', marker="x")
# plt.scatter(vy_max, vx_max, color='green', marker='x', label='max energy')
plt.ylabel("vx (normalized)")
plt.xlabel("vy (normalized)")

In [ ]:
np.array([velstoprobe[det_idxs[:, 0]], velstoprobe[det_idxs[:, 1]]]).T

In [ ]:
plt.imshow(det)

In [ ]:
energies.shape

In [ ]:
velocities_grid = np.meshgrid(velstoprobe, velstoprobe, indexing="ij")

In [ ]:
velocities_grid

In [ ]:
argmax_idx = np.unravel_index(np.argmax(energies), energies.shape)
velocities_grid[0][argmax_idx], velocities_grid[1][argmax_idx]

In [ ]:
vel1norm

In [ ]:
np.unravel_index(np.argmax(energies), energies.shape)

In [ ]:
from matplotlib.colors import LogNorm

plt.imshow(np.abs(np.fft.fftshift(F[:, 0, :])), origin="lower", extent=(fx.min(), fx.max(), ft.min(), ft.max()), norm=LogNorm())
tt = np.linspace(-10, 10, 100)
y = -tt * vel1norm[0]
plt.plot(tt, y)
plt.xlim(-20, 20)
plt.ylim(-20, 20)
plt.colorbar()

In [ ]:
"""
3D Fourier Spectrum Visualizer using Plotly
Supports both scatter plot and voxel (isosurface/volume) visualization.
Transparency and color are driven by log magnitude of Fourier coefficients.
"""

import numpy as np
import plotly.graph_objects as go


def plot_fourier_spectrum_3d(
    spectrum: np.ndarray,
    mode: str = "scatter",
    threshold_percentile: float = 80.0,
    colorscale: str = "Viridis",
    title: str = "3D Fourier Spectrum",
) -> go.Figure:
    """
    Visualize a 3D Fourier spectrum using Plotly.

    Parameters
    ----------
    spectrum : np.ndarray
        Complex 3D array (output of np.fft.fftn or similar).
        Shape: (Nx, Ny, Nz)
    mode : str
        'scatter' — scatter plot where dot size & opacity encode log magnitude.
        'voxel'   — volume/isosurface rendering where opacity encodes log magnitude.
    threshold_percentile : float
        Only points/voxels above this percentile of log-magnitude are shown.
        Useful to declutter the plot (default: 80 → top 20% shown).
    colorscale : str
        Plotly colorscale name (e.g. 'Viridis', 'Plasma', 'Turbo', 'Hot').
    title : str
        Figure title.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    if spectrum.ndim != 3:
        raise ValueError(f"Expected a 3D array, got shape {spectrum.shape}")

    Nx, Ny, Nt = spectrum.shape

    # ── Frequency axes (fftshift so DC is at centre) ──────────────────────────
    fx = np.fft.fftshift(np.fft.fftfreq(Nx, d=1.0)) * Nx   # cycles, step = 1
    fy = np.fft.fftshift(np.fft.fftfreq(Ny, d=1.0)) * Ny
    ft = np.fft.fftshift(np.fft.fftfreq(Nt, d=1.0)) * Nt

    # ── Shift spectrum so DC is at centre ─────────────────────────────────────
    shifted = np.fft.fftshift(spectrum)

    # ── Log magnitude ─────────────────────────────────────────────────────────
    magnitude = np.abs(shifted)
    log_mag = np.log1p(magnitude)           # log(1 + |F|) — safe for zeros

    # ── Threshold mask ────────────────────────────────────────────────────────
    thresh = np.percentile(log_mag, threshold_percentile)
    mask = log_mag >= thresh

    # ── Build coordinate grids ────────────────────────────────────────────────
    FX, FY, FT = np.meshgrid(fx, fy, ft, indexing="ij")

    # ── Normalised log-magnitude [0, 1] for colour / opacity / size ───────────
    lm_flat = log_mag[mask]
    lm_min, lm_max = lm_flat.min(), lm_flat.max()
    lm_norm = (lm_flat - lm_min) / (lm_max - lm_min + 1e-12)

    if mode == "scatter":
        fig = _scatter_plot(
            FX[mask], FY[mask], FT[mask],
            lm_norm, lm_flat,
            colorscale,
        )
    elif mode == "voxel":
        fig = _volume_plot(
            FX, FY, FT, log_mag,
            thresh, colorscale,
        )
    else:
        raise ValueError(f"mode must be 'scatter' or 'voxel', got '{mode}'")

    # ── Axis labels / limits ──────────────────────────────────────────────────
    half = (
        np.array([Nx, Ny, Nt]) // 2
    )
    ax_range = lambda h: [-(h), h]

    fig.update_layout(
        title=dict(text=title, font=dict(size=18)),
        scene=dict(
            xaxis=dict(
                title="Frequency X (step=1)",
                range=ax_range(half[0]),
            ),
            yaxis=dict(
                title="Frequency Y (step=1)",
                range=ax_range(half[1]),
            ),
            zaxis=dict(
                title="Frequency T (step=1)",
                range=ax_range(half[2]),
            ),
            bgcolor="rgb(10,10,20)",
            xaxis_backgroundcolor="rgb(15,15,30)",
            yaxis_backgroundcolor="rgb(15,15,30)",
            zaxis_backgroundcolor="rgb(15,15,30)",
        ),
        paper_bgcolor="rgb(10,10,20)",
        font=dict(color="white"),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    return fig


# ── Scatter helper ────────────────────────────────────────────────────────────

def _scatter_plot(x, y, z, lm_norm, lm_raw, colorscale):
    """Scatter3d: colour, opacity, and marker size all scale with log magnitude."""
    # Marker size: map [0,1] → [2, 20] px
    sizes = 1 + (lm_norm ** 2) * 19

    # Per-point opacity: map [0,1] → [0.15, 1.0]
    opacities = 0.15 + lm_norm * 0.85

    # Plotly Scatter3d needs a single opacity scalar; we encode it via the
    # alpha channel by building an rgba colour array.
    import plotly.colors as pc

    # Sample the colorscale at each normalised value
    rgb_list = pc.sample_colorscale(colorscale, lm_norm.tolist())

    def rgba(rgb_str, alpha):
        """'rgb(r,g,b)' → 'rgba(r,g,b,a)'"""
        inner = rgb_str[4:-1]          # strip 'rgb(' and ')'
        return f"rgba({inner},{alpha:.3f})"

    colors = [rgba(rgb, a) for rgb, a in zip(rgb_list, opacities)]

    trace = go.Scatter3d(
        x=x.ravel(), y=y.ravel(), z=z.ravel(),
        mode="markers",
        marker=dict(
            size=sizes,
            color=lm_raw,               # drives the colorbar
            colorscale=colorscale,
            showscale=True,
            colorbar=dict(title="log(1+|F|)"),
            opacity=1.0,                # override below via line trick
            line=dict(width=0),
        ),
        # Override colours with per-point rgba for true per-point alpha
        # Plotly supports this via marker.color as a list of rgba strings
        text=[f"log|F|={v:.3f}" for v in lm_raw],
        hovertemplate="x=%{x:.1f}  y=%{y:.1f}  z=%{z:.1f}<br>%{text}<extra></extra>",
    )
    # Apply per-point rgba colours (overrides colorscale but keeps colorbar)
    trace.marker.color = colors
    trace.marker.colorscale = None
    trace.marker.showscale = False

    fig = go.Figure(data=[trace])
    return fig


# ── Volume helper ─────────────────────────────────────────────────────────────

def _volume_plot(FX, FY, FZ, log_mag, thresh, colorscale):
    """
    Volume rendering: opacity proportional to log magnitude.
    Uses go.Volume for smooth voxel-like rendering.
    """
    lm_min = log_mag.min()
    lm_max = log_mag.max()

    # go.Volume expects flattened 1-D arrays
    trace = go.Volume(
        x=FX.ravel(),
        y=FY.ravel(),
        z=FZ.ravel(),
        value=log_mag.ravel(),
        isomin=thresh,
        isomax=lm_max,
        opacity=0.15,           # base opacity (blended per voxel by Plotly)
        surface_count=20,       # number of isosurface layers → more = smoother
        colorscale=colorscale,
        showscale=True,
        colorbar=dict(title="log(1+|F|)"),
        caps=dict(x_show=False, y_show=False, z_show=False),
    )
    fig = go.Figure(data=[trace])
    return fig


# ── Velocity plane ────────────────────────────────────────────────────────────

def plot_velocity_plane(
    spectrum: np.ndarray,
    vx: float,
    vy: float,
    colorscale: str = "Hot",
    title: str | None = None,
    plane_opacity: float = 0.18,
    plane_color: str = "rgba(80,160,255,0.18)",
) -> go.Figure:
    """
    Overlay the constraint plane  vx·fx + vy·fy + ft = 0  on the 3-D Fourier
    spectrum scatter plot.

    The spectrum axes are  (fx, fy, ft)  where the third axis is *time*
    frequency.  Shape of ``spectrum``:  (Nx, Ny, Nt).

    The plane passes through the origin with normal  n = (vx, vy, 1).
    We render it as a ``go.Surface`` mesh spanning the visible frequency box,
    and add the spectrum scatter on top so you can see which coefficients lie
    on (or near) the plane.

    Parameters
    ----------
    spectrum : np.ndarray, complex, shape (Nx, Ny, Nt)
    vx, vy   : float  — spatial velocity components (normalised units)
    colorscale : Plotly colorscale for the spectrum scatter
    title    : figure title (auto-generated if None)
    plane_opacity : float in [0,1] — how transparent the plane surface is
    plane_color   : CSS rgba string for the plane mesh fill

    Returns
    -------
    plotly.graph_objects.Figure
    """
    if spectrum.ndim != 3:
        raise ValueError(f"Expected shape (Nx, Ny, Nt), got {spectrum.shape}")

    Nx, Ny, Nt = spectrum.shape

    # ── Frequency axes, shifted so DC is at centre ────────────────────────────
    fx = np.fft.fftshift(np.fft.fftfreq(Nx, d=1.0)) * Nx
    fy = np.fft.fftshift(np.fft.fftfreq(Ny, d=1.0)) * Ny
    ft = np.fft.fftshift(np.fft.fftfreq(Nt, d=1.0)) * Nt

    shifted   = np.fft.fftshift(spectrum)
    magnitude = np.abs(shifted)
    log_mag   = np.log1p(magnitude)

    FX, FY, FT = np.meshgrid(fx, fy, ft, indexing="ij")

    # ── Scatter: show all points, opacity/size/colour by log-magnitude ─────────
    lm_all  = log_mag.ravel()
    lm_min, lm_max = lm_all.min(), lm_all.max()
    lm_norm = (lm_all - lm_min) / (lm_max - lm_min + 1e-12)

    sizes     = 1.5 + lm_norm * 10
    opacities = 0.05 + lm_norm * 0.75

    import plotly.colors as pc
    rgb_list = pc.sample_colorscale(colorscale, lm_norm.tolist())

    def rgba(rgb_str, alpha):
        inner = rgb_str[4:-1]
        return f"rgba({inner},{alpha:.3f})"

    colors = [rgba(r, a) for r, a in zip(rgb_list, opacities)]

    scatter = go.Scatter3d(
        x=FX.ravel(), y=FY.ravel(), z=FT.ravel(),
        mode="markers",
        name="Spectrum",
        marker=dict(size=sizes, color=colors, line=dict(width=0)),
        hovertemplate=(
            "fx=%{x:.1f}  fy=%{y:.1f}  ft=%{z:.1f}<br>"
            "log|F|=%{text}<extra></extra>"
        ),
        text=[f"{v:.3f}" for v in lm_all],
    )

    # ── Plane  ft = -(vx·fx + vy·fy) ─────────────────────────────────────────
    # Span the plane over the visible fx/fy box; clip ft to the ft axis range.
    hx, hy, ht = Nx // 2, Ny // 2, Nt // 2
    px = np.linspace(-hx, hx, 60)
    py = np.linspace(-hy, hy, 60)
    PX, PY = np.meshgrid(px, py)
    PT = -(vx * PX + vy * PY)
    PT = np.clip(PT, -ht, ht)          # keep inside the rendered volume

    plane = go.Surface(
        x=PX, y=PY, z=PT,
        opacity=plane_opacity,
        colorscale=[[0, plane_color], [1, plane_color]],
        showscale=False,
        name=f"Plane  {vx:+.2f}·fx + {vy:+.2f}·fy + ft = 0",
        hoverinfo="skip",
        contours=dict(
            x=dict(show=True, color="rgba(120,180,255,0.35)", width=1),
            y=dict(show=True, color="rgba(120,180,255,0.35)", width=1),
        ),
    )

    # ── Points on/near the plane ───────────────────────────────────────────────
    # Distance from each (fx, fy, ft) grid point to the plane, measured along ft
    residual = np.abs(FT + vx * FX + vy * FY)   # |ft + vx·fx + vy·fy|
    tol      = max(1.0, 0.5 * min(Nx, Ny, Nt) / 16)   # ≈ half a step
    on_plane = residual <= tol

    lm_plane  = log_mag[on_plane]
    lm_p_norm = (lm_plane - lm_min) / (lm_max - lm_min + 1e-12)
    sizes_p   = 3 + lm_p_norm * 14
    opac_p    = 0.5 + lm_p_norm * 0.5
    rgb_p     = pc.sample_colorscale("Plasma", lm_p_norm.tolist())
    colors_p  = [rgba(r, a) for r, a in zip(rgb_p, opac_p)]

    on_scatter = go.Scatter3d(
        x=FX[on_plane], y=FY[on_plane], z=FT[on_plane],
        mode="markers",
        name="On plane",
        marker=dict(size=sizes_p, color=colors_p, line=dict(width=0)),
        hovertemplate=(
            "fx=%{x:.1f}  fy=%{y:.1f}  ft=%{z:.1f}<br>"
            "log|F|=%{text}<extra></extra>"
        ),
        text=[f"{v:.3f}" for v in lm_plane],
    )

    # ── Layout ────────────────────────────────────────────────────────────────
    auto_title = (
        title if title else
        f"Velocity plane  {vx:+.2f}·f_x + {vy:+.2f}·f_y + f_t = 0"
    )
    # fig = go.Figure(data=[scatter, plane, on_scatter])
    fig = go.Figure(data=[plane])
    fig.update_layout(
        title=dict(text=auto_title, font=dict(size=17)),
        legend=dict(
            x=0.01, y=0.99,
            bgcolor="rgba(20,20,40,0.7)",
            font=dict(color="white", size=11),
        ),
        scene=dict(
            xaxis=dict(title="f_x  (step=1)", range=[-hx, hx]),
            yaxis=dict(title="f_y  (step=1)", range=[-hy, hy]),
            zaxis=dict(title="f_t  (step=1)", range=[-ht, ht]),
            bgcolor="rgb(10,10,20)",
            xaxis_backgroundcolor="rgb(15,15,30)",
            yaxis_backgroundcolor="rgb(15,15,30)",
            zaxis_backgroundcolor="rgb(15,15,30)",
            camera=dict(eye=dict(x=1.6, y=1.4, z=1.0)),
        ),
        paper_bgcolor="rgb(10,10,20)",
        font=dict(color="white"),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    return fig

In [ ]:
fig = plot_fourier_spectrum_3d(
    F.T,
    mode="scatter",
    threshold_percentile=90.0,
    colorscale="Viridis",
    title="3D Fourier Spectrum (Scatter)",
)
# Add to an existing figure as extra traces:
planes = []
for v in result.detected_velocities:
    vx, vy = v
    vx = normalize_velocities(vx, Nt, Nx)
    vy = normalize_velocities(vy, Nt, Ny)
    plane_fig = plot_velocity_plane(F.T, vx=vx, vy=vy)
    planes.append(plane_fig.data[0])  # extract the plane trace
    fig.add_trace(plane_fig.data[0])
# save as html
fig.write_html(f"../results/{dataname}_fourier_spectrum_3d_scatter.html")

In [ ]:
F

In [ ]:
from matplotlib.colors import LogNorm
plt.imshow(np.abs(np.fft.fftshift(F[:, :, 0])), norm=LogNorm())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import shift, gaussian_filter
from matplotlib.colors import Normalize

# ── helpers ──────────────────────────────────────────────────────────────────

def make_video(width, height, n_frames, objects):
    """
    Create a synthetic grayscale video.

    objects: list of dicts with keys
        cx, cy   – starting centre (pixels)
        vx, vy   – velocity in pixels/frame
        r        – radius
        intensity
    """
    video = np.zeros((n_frames, height, width), dtype=np.float64)
    yy, xx = np.mgrid[0:height, 0:width]
    for t in range(n_frames):
        for obj in objects:
            cx = obj["cx"] + t * obj["vx"]
            cy = obj["cy"] + t * obj["vy"]
            mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= obj["r"] ** 2
            video[t][mask] += obj["intensity"]
    return np.clip(video, 0, 1)


# ── Approach 1 : Phase-based segmentation ────────────────────────────────────

def gabor_filter_bank(video, fx, fy, sigma_x=4.0, sigma_y=4.0):
    """
    Apply a complex Gabor filter tuned to spatial frequency (fx, fy)
    (in cycles/pixel) to every frame independently.

    Returns complex-valued response array of shape (T, H, W).
    """
    T, H, W = video.shape
    # Build kernel in frequency domain (faster than spatial conv for large σ)
    v = np.fft.fftfreq(H)
    u = np.fft.fftfreq(W)
    UU, VV = np.meshgrid(u, v)

    # Gaussian centred on (fx, fy) in frequency domain → Gabor in space
    kernel_fft = np.exp(
        -0.5 * (((UU - fx) / (1 / (2 * np.pi * sigma_x))) ** 2
                + ((VV - fy) / (1 / (2 * np.pi * sigma_y))) ** 2)
    )

    responses = np.zeros((T, H, W), dtype=np.complex128)
    for t in range(T):
        frame_fft = np.fft.fft2(video[t])
        responses[t] = np.fft.ifft2(frame_fft * kernel_fft)
    return responses


def phase_segmentation(video, vx, vy,
                        fx=0.1, fy=0.0,
                        sigma_x=4.0, sigma_y=4.0,
                        phase_tol=0.3,
                        smooth_sigma=2.0):
    """
    Segment pixels whose inter-frame phase advance matches (vx, vy).

    Parameters
    ----------
    vx, vy      velocity in pixels/frame
    fx, fy      spatial frequency (cycles/pixel) of the Gabor probe
    phase_tol   tolerance in radians around the expected phase
    smooth_sigma spatial smoothing of the score map before thresholding

    Returns
    -------
    score  (T-1, H, W) – phase-match score ∈ [0,1] per frame pair
    mask   (T-1, H, W) – binary segmentation
    """
    # Expected phase advance per frame at this spatial frequency
    expected_phase = 2 * np.pi * (fx * vx + fy * vy)

    responses = gabor_filter_bank(video, fx, fy, sigma_x, sigma_y)

    T, H, W = video.shape
    scores = np.zeros((T - 1, H, W))
    for t in range(T - 1):
        # Inter-frame phase difference
        phase_diff = np.angle(responses[t + 1] * np.conj(responses[t]))
        # Score: cosine similarity to expected phase (∈ [-1, 1] → rescale to [0,1])
        scores[t] = 0.5 * (np.cos(phase_diff - expected_phase) + 1)

    if smooth_sigma > 0:
        scores = gaussian_filter(scores, sigma=[0, smooth_sigma, smooth_sigma])

    mask = scores > 0.75   # threshold
    return scores, mask


# ── Approach 2 : Motion-compensated accumulation ─────────────────────────────

def motion_compensated_segmentation(video, vx, vy,
                                     diff_threshold=0.15,
                                     smooth_sigma=2.0):
    """
    Warp each frame by -t*(vx, vy) so the target object becomes still,
    then segment by comparing per-frame to the stabilised mean.

    Returns
    -------
    stabilised  (T, H, W) – motion-compensated frames
    mean_frame  (H, W)    – temporal mean of stabilised video
    residuals   (T, H, W) – |frame - mean| in stabilised space
    mask        (T, H, W) – binary: 1 where residual is LOW (i.e. object is stable)
    """
    T, H, W = video.shape
    stabilised = np.zeros_like(video)

    for t in range(T):
        # Shift opposite to motion so accumulated displacement cancels
        dy = -t * vy
        dx = -t * vx
        stabilised[t] = shift(video[t], shift=[dy, dx], mode="wrap")

    mean_frame = stabilised.mean(axis=0)

    residuals = np.abs(stabilised - mean_frame[np.newaxis])
    if smooth_sigma > 0:
        residuals = gaussian_filter(residuals,
                                    sigma=[0, smooth_sigma, smooth_sigma])

    # Low residual → pixel is stable across compensated frames → moving object
    mask = residuals < diff_threshold

    return stabilised, mean_frame, residuals, mask


def fourier_plane_segmentation(video, vx, vy, plane_thickness=1.5, smooth=3.0):
    """
    Keep only 3D Fourier coefficients near the plane fx·vx + fy·vy + ft = 0,
    inverse transform, and threshold the recovered energy.

    plane_thickness: in units of the frequency bin spacing
    """
    T, H, W = video.shape

    # 3D FFT — axes order: (ft, fy, fx)
    V = np.fft.fftn(video)

    ft = np.fft.fftfreq(T)     # shape (T,)
    fy = np.fft.fftfreq(H)     # shape (H,)
    fx = np.fft.fftfreq(W)     # shape (W,)

    FT, FY, FX = np.meshgrid(ft, fy, fx, indexing="ij")  # (T,H,W)

    # Signed distance to the plane (in frequency units)
    # Plane normal is (vx, vy, 1) (unnormalised)
    plane_val = FX * vx + FY * vy + FT
    normal_mag = np.sqrt(vx**2 + vy**2 + 1.0)
    dist = np.abs(plane_val) / normal_mag          # true perpendicular distance

    # Bin spacing (smallest nonzero frequency step)
    df = 1.0 / max(T, H, W)
    thickness_freq = plane_thickness * df

    # Soft Gaussian window instead of hard cut (avoids ringing)
    window = np.exp(-0.5 * (dist / thickness_freq) ** 2)

    # Zero out DC — it's on every plane and carries no motion information
    V_filtered = V * window
    V_filtered[0, 0, 0] = 0

    # Reconstruct
    recon = np.real(np.fft.ifftn(V_filtered))

    # Energy map
    energy = recon ** 2
    if smooth > 0:
        energy = gaussian_filter(energy, sigma=[0, smooth, smooth])

    # Normalise per frame for display
    e_min, e_max = energy.min(), energy.max()
    energy_norm = (energy - e_min) / (e_max - e_min + 1e-12)

    threshold = np.percentile(energy_norm, 85)
    mask = energy_norm > threshold

    return recon, energy_norm, mask


def print_fourier_sanity(video, vx, vy, W, H, T):
    """
    Show that energy of the moving object clusters near the plane
    fx*vx + fy*vy + ft = 0  in the 3-D Fourier domain.
    """
    V = np.fft.fftn(video)   # shape (T, H, W) → axes: ft, fy, fx
    power = np.abs(V) ** 2

    ft_ax = np.fft.fftfreq(T)
    fy_ax = np.fft.fftfreq(H)
    fx_ax = np.fft.fftfreq(W)

    FT, FY, FX = np.meshgrid(ft_ax, fy_ax, fx_ax, indexing="ij")
    plane_dist = np.abs(FX * vx + FY * vy + FT)   # distance to motion plane

    on_plane  = power[plane_dist < 0.02].sum()
    off_plane = power[plane_dist >= 0.02].sum()
    print(f"\n3-D Fourier sanity check (vx={vx}, vy={vy}):")
    print(f"  Energy on motion plane  : {on_plane:.3e}")
    print(f"  Energy off motion plane : {off_plane:.3e}")
    print(f"  Fraction on plane       : {on_plane / (on_plane + off_plane):.3f}")

In [ ]:
import warnings
from typing import List, Tuple

class AdditiveImageSeparator:
    """
    Separate m additive component images from N composite frames.
 
    Each component image is assumed to translate at a distinct, known,
    constant velocity (pixels/frame).  Works for any m >= 2, and any N >= m
    (more frames gives better noise robustness).
 
    The book (Vernon 2001, Ch.3) states that for m components you need 2m
    equations (i.e. 2m frames) when velocities are UNKNOWN.  Here velocities
    are assumed already found (e.g. via Hough transform), so the unknowns
    are only the m phasors Fⁱ_t0 — making N >= m sufficient, with N >= 2m
    still recommended for conditioning.
 
    Parameters
    ----------
    velocities : list of (vx, vy) tuples, length m
        Known velocity of each component in pixels/frame.
    dt : float
        Time step between frames (default 1.0).
    eps : float
        Tikhonov regularisation for near-singular (degenerate) frequencies.
 
    Example
    -------
    sep = AdditiveImageSeparator(velocities=[(2, 1), (-1, 2), (0, -3)])
    imgs = sep.separate(frames)   # frames: (N, H, W) array
    img1, img2, img3 = imgs
    """
 
    def __init__(
        self,
        velocities: List[Tuple[float, float]],
        dt: float = 1.0,
        eps: float = 1e-8,
    ):
        self.velocities = [np.array(v, dtype=float) for v in velocities]
        self.m = len(velocities)
        self.dt = dt
        self.eps = eps
 
    # ── public ────────────────────────────────────────────────────────────────
 
    def separate(self, frames: np.ndarray) -> np.ndarray:
        """
        Recover all N frames for each of the m component images.
 
        Parameters
        ----------
        frames : (N, H, W) float array
            Composite image sequence.  frames[j] = sum_i f^i_tj.
 
        Returns
        -------
        sequences : (m, N, H, W) float array
            sequences[i, j] is component i at time t_j.
            sequences[i, 0] is the t=0 frame (same as before).
        """
        frames = np.asarray(frames, dtype=float)
        N, H, W = frames.shape
        m = self.m
 
        if N < m:
            raise ValueError(
                f"Need at least N={m} frames for m={m} components; got {N}. "
                f"Recommended N >= 2m = {2*m} for good conditioning."
            )
        if N < 2 * m:
            warnings.warn(
                f"N={N} < 2m={2*m}: fewer frames than the book recommends. "
                "Results may be inaccurate. Add more frames if possible.",
                stacklevel=2,
            )
 
        # ── 1. FFT every frame  →  shape (N, H, W) complex ────────────────
        F = np.stack([np.fft.fft2(frames[j]) for j in range(N)], axis=0)
 
        # ── 2. Angular spatial-frequency grids (FFT layout) ───────────────
        wx = 2 * np.pi * np.fft.fftfreq(W)   # (W,)
        wy = 2 * np.pi * np.fft.fftfreq(H)   # (H,)
        WX, WY = np.meshgrid(wx, wy)           # (H, W)
        HW = H * W
 
        # ── 3. Per-frame phase factors for each component ─────────────────
        #   ΔΦᵢ(wx,wy) = exp(-i (wx·vxᵢ + wy·vyᵢ) · dt)
        #   shape (HW,) per component
        dPhi = []
        for v in self.velocities:
            phi = np.exp(-1j * (WX * v[0] + WY * v[1]) * self.dt).ravel()
            dPhi.append(phi)                   # (HW,)
 
        # ── 4. Design matrix A  (Vandermonde in ΔΦ) ──────────────────────
        #   A[j, i, freq] = ΔΦᵢ(freq)^j
        #   We build shape (HW, N, m) — frequencies as the batch dimension.
        j_idx = np.arange(N)                   # (N,)
 
        # Each column: (HW,)^j broadcast → (N, HW), then transposed → (HW, N)
        cols = [(dPhi[i][np.newaxis, :] ** j_idx[:, np.newaxis]).T
                for i in range(m)]             # each (HW, N)
 
        A = np.stack(cols, axis=-1)            # (HW, N, m)
 
        # ── 5. Observation vector b  →  (HW, N, 1) ────────────────────────
        b = F.reshape(N, HW).T[:, :, np.newaxis]  # (HW, N, 1)
 
        # ── 6. Least-squares via normal equations: x = (AᴴA)⁻¹ Aᴴb ───────
        #   Solves for Fⁱ_t0 at every (wx,wy) simultaneously.
        #   AᴴA : (HW, m, m)
        #   Aᴴb : (HW, m, 1)
        AH = np.conj(A).transpose(0, 2, 1)    # (HW, m, N)
        AHA = AH @ A                           # (HW, m, m)
        AHb = AH @ b                           # (HW, m, 1)
 
        # Tikhonov regularisation: AᴴA + ε·I
        eye_m = np.eye(m, dtype=complex)[np.newaxis]  # (1, m, m)
        AHA += self.eps * eye_m
 
        x = np.linalg.solve(AHA, AHb)         # (HW, m, 1)
        x = x.squeeze(-1)                      # (HW, m)
 
        # x[:, i] = Fⁱ_t0(wx, wy)  — the t=0 Fourier spectrum of component i
 
        # ── 7. Propagate each component forward: Fⁱ_tj = Fⁱ_t0 · ΔΦᵢ^j ──
        #   then iFFT to get all N spatial frames per component.
        #
        #   sequences[i, j] = ifft2( Fⁱ_t0 · ΔΦᵢ^j )
        sequences = np.empty((m, N, H, W), dtype=float)
 
        for i in range(m):
            Fi_t0 = x[:, i].reshape(H, W)          # (H, W) complex
            dPhi_i = dPhi[i].reshape(H, W)          # (H, W) complex
 
            for j in range(N):
                # Apply j-th power of the phase ramp → shift component by j frames
                Fi_tj = Fi_t0 * (dPhi_i ** j)       # (H, W) complex
                sequences[i, j] = np.real(np.fft.ifft2(Fi_tj))
 
        return sequences

In [ ]:
result.detected_velocities

In [ ]:
# ais = AdditiveImageSeparator(velocities=[*(result.detected_velocities[[1]])])
# # ais = AdditiveImageSeparator(velocities=[(0, 0)])
# # ais = AdditiveImageSeparator(velocities=[vel1norm, vel2norm, (0, 0)])
# separated_imgs = ais.separate(fluxgrid)
# for i, sep_imgs in enumerate(separated_imgs):
#     to_video(sep_imgs, f"../results/separated_component_{i}.mp4", playback_fps=30, cmap="gray")

In [ ]:
for i, vel in enumerate([*result.detected_velocities, (0, 0)]):
    ais = AdditiveImageSeparator(velocities=[vel])
    separated_imgs = ais.separate(fluxgrid)
    to_video(separated_imgs[0], f"../results/{dataname}_component_{i}_vx{vel[0]:.2f}_vy{vel[1]:.2f}.mp4", playback_fps=30, cmap="gray")

In [ ]:
# Target object: moves at (vx=3, vy=1) px/frame
# Background blob: stationary
VX, VY = vel1norm[0], vel1norm[1]
# VX, VY = vel2norm[0], vel2norm[1]


# ── Phase segmentation ──────────────────────────────────────────────────
# Probe at fx = 1/(2r) ≈ dominant spatial frequency of the blob
probe_fx = 1.0 / (2 * 12)
phase_scores, phase_mask = phase_segmentation(
    fluxgrid, VX, VY,
    fx=probe_fx, fy=probe_fx,
    sigma_x=3.0, sigma_y=3.0,
    phase_tol=0.3,
    smooth_sigma=2.0,
)

# ── Motion-compensated segmentation ────────────────────────────────────
stabilised, mean_frame, residuals, mc_mask = motion_compensated_segmentation(
    fluxgrid, VX, VY, diff_threshold=0.2, smooth_sigma=2.0
)

# ── Fourier plane segmentation ─────────────────────────────────────────
plane_recon, plane_energy, plane_mask = fourier_plane_segmentation(
    fluxgrid, VX, VY, plane_thickness=2.0, smooth=3.0
)

# ── Visualise ───────────────────────────────────────────────────────────
frame_idx = Nt // 2   # which frame to show
fig, axes = plt.subplots(3, 4, figsize=(15, 11))
fig.suptitle("Motion Segmentation — two approaches", fontsize=14, fontweight="bold")

def show(ax, img, title, cmap="gray", vmin=0, vmax=1):
    ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax, origin="upper",
                interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.axis("off")

# Row 0 – raw video
show(axes[0, 0], fluxgrid[0],          "Frame 0 (raw)")
show(axes[0, 1], fluxgrid[frame_idx],  f"Frame {frame_idx} (raw)")
show(axes[0, 2], fluxgrid[-1],         "Frame –1 (raw)")
show(axes[0, 3], fluxgrid.mean(0),     "Temporal mean (raw)")

# Row 1 – phase-based
show(axes[1, 0], phase_scores[frame_idx - 1], "Phase score",  cmap="hot")
show(axes[1, 1], phase_mask[frame_idx - 1].astype(float), "Phase mask")
# overlay on frame
overlay_ph = np.stack([fluxgrid[frame_idx]] * 3, axis=-1)
overlay_ph[..., 0] = np.where(phase_mask[frame_idx - 1],
                                np.clip(fluxgrid[frame_idx] + 0.4, 0, 1),
                                fluxgrid[frame_idx])
axes[1, 2].imshow(overlay_ph, origin="upper", interpolation="nearest")
axes[1, 2].set_title("Phase mask overlay", fontsize=9); axes[1, 2].axis("off")
show(axes[1, 3], phase_scores.mean(0), "Phase score (mean t)", cmap="hot")

# Row 2 – motion-compensated
show(axes[2, 0], stabilised[frame_idx], f"Stabilised frame {frame_idx}")
show(axes[2, 1], mean_frame,            "Stabilised mean")
show(axes[2, 2], residuals[frame_idx],  "Residual",  cmap="hot", vmax=0.5)
show(axes[2, 3], mc_mask[frame_idx].astype(float), "MC mask")

plt.tight_layout()
plt.savefig(f"../results/{dataname}_motion_seg_demo.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved → {dataname}_motion_seg_demo.png")

# ── 3-D Fourier sanity check ────────────────────────────────────────────
# print_fourier_sanity(video, VX, VY, W, H, T)

In [ ]:
to_video(phase_mask, "../results/phase_mask_video.mp4", playback_fps=30, cmap="gray")
to_video(stabilised, "../results/stabilised_video.mp4", playback_fps=30, cmap="gray")
to_video(mc_mask, "../results/mc_mask_video.mp4", playback_fps=30, cmap="gray")
to_video(plane_mask, "../results/plane_mask_video.mp4", playback_fps=30, cmap="gray")
to_video(plane_recon, "../results/plane_recon_video.mp4", playback_fps=30, cmap="gray")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import shift, gaussian_filter


def make_video(width, height, n_frames, objects, noise=0.02):
    video = np.zeros((n_frames, height, width), dtype=np.float64)
    yy, xx = np.mgrid[0:height, 0:width]
    for t in range(n_frames):
        for obj in objects:
            cx = obj["cx"] + t * obj["vx"]
            cy = obj["cy"] + t * obj["vy"]
            mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= obj["r"] ** 2
            video[t][mask] += obj["intensity"]
    if noise > 0:
        video += np.random.randn(*video.shape) * noise
    return np.clip(video, 0, 1)


# ── Fix 1: smart probe selection ─────────────────────────────────────────────

def best_probe_frequency(vx, vy, target_phase=np.pi / 2, n_probes=8):
    """
    Return a list of (fx, fy) probe frequencies such that
        2π (fx·vx + fy·vy) ≈ target_phase
    spread over multiple orientations so at least one has strong
    spatial energy in any object.
    """
    speed = np.sqrt(vx**2 + vy**2)
    if speed == 0:
        return [(0.1, 0.0)]

    # Direction of velocity and perpendicular
    ux, uy = vx / speed, vy / speed

    # The magnitude of f along velocity direction that gives target_phase:
    # 2π · f_parallel · speed = target_phase  →  f_parallel = target_phase/(2π·speed)
    f_parallel = target_phase / (2 * np.pi * speed)

    probes = []
    # Vary the perpendicular component so we cover different orientations
    for k in np.linspace(-1, 1, n_probes):
        f_perp = k * f_parallel          # orthogonal component (doesn't affect phase)
        fx = f_parallel * ux - f_perp * uy
        fy = f_parallel * uy + f_perp * ux
        # Keep within Nyquist
        if abs(fx) < 0.5 and abs(fy) < 0.5:
            probes.append((fx, fy))
    return probes


def gabor_response(video, fx, fy, sigma_cycles=2.0):
    """Complex Gabor filter response. sigma_cycles controls bandwidth."""
    T, H, W = video.shape
    u = np.fft.fftfreq(W)
    v = np.fft.fftfreq(H)
    UU, VV = np.meshgrid(u, v)
    # Bandwidth in frequency units
    sigma_u = 1.0 / (2 * np.pi * sigma_cycles / max(abs(fx), abs(fy), 1e-6)) if fx != 0 or fy != 0 else 0.05
    kernel = np.exp(-0.5 * (((UU - fx) ** 2 + (VV - fy) ** 2) / sigma_u ** 2))
    responses = np.zeros((T, H, W), dtype=np.complex128)
    for t in range(T):
        responses[t] = np.fft.ifft2(np.fft.fft2(video[t]) * kernel)
    return responses


# ── Fix 2: discriminative phase score ────────────────────────────────────────

def phase_score_discriminative(video, vx, vy, sigma_cycles=2.0, smooth=2.0):
    """
    For each probe, score = cos(Δφ - φ_expected) - cos(Δφ - 0)
    i.e. how much MORE consistent with the target velocity than with stationary.
    Aggregate over probe bank by taking the max.
    """
    probes = best_probe_frequency(vx, vy, target_phase=np.pi / 2)
    T, H, W = video.shape
    agg_score = np.full((T - 1, H, W), -np.inf)

    for fx, fy in probes:
        phi_expected = 2 * np.pi * (fx * vx + fy * vy)
        if abs(phi_expected) < 0.1:          # skip degenerate probes
            continue
        resp = gabor_response(video, fx, fy, sigma_cycles)
        amp  = np.abs(resp)                  # only score where filter has response

        for t in range(T - 1):
            delta_phi = np.angle(resp[t + 1] * np.conj(resp[t]))
            # How well does this match target vs stationary?
            score_target     = np.cos(delta_phi - phi_expected)
            score_stationary = np.cos(delta_phi)               # expected=0 for bg
            discriminative   = score_target - score_stationary  # ∈ [-2, 2]
            # Weight by filter amplitude (ignore low-energy regions)
            weighted = discriminative * (amp[t] / (amp[t].max() + 1e-9))
            agg_score[t] = np.maximum(agg_score[t], weighted)

    if smooth > 0:
        agg_score = gaussian_filter(agg_score, sigma=[0, smooth, smooth])
    return agg_score


# ── Fix 4: 3D Fourier plane filter (most robust) ─────────────────────────────

def fourier_plane_segmentation(video, vx, vy, plane_thickness=1.5, smooth=3.0):
    """
    Keep only 3D Fourier coefficients near the plane fx·vx + fy·vy + ft = 0,
    inverse transform, and threshold the recovered energy.

    plane_thickness: in units of the frequency bin spacing
    """
    T, H, W = video.shape

    # 3D FFT — axes order: (ft, fy, fx)
    V = np.fft.fftn(video)

    ft = np.fft.fftfreq(T)     # shape (T,)
    fy = np.fft.fftfreq(H)     # shape (H,)
    fx = np.fft.fftfreq(W)     # shape (W,)

    FT, FY, FX = np.meshgrid(ft, fy, fx, indexing="ij")  # (T,H,W)

    # Signed distance to the plane (in frequency units)
    # Plane normal is (vx, vy, 1) (unnormalised)
    plane_val = FX * vx + FY * vy + FT
    normal_mag = np.sqrt(vx**2 + vy**2 + 1.0)
    dist = np.abs(plane_val) / normal_mag          # true perpendicular distance

    # Bin spacing (smallest nonzero frequency step)
    df = 1.0 / max(T, H, W)
    thickness_freq = plane_thickness * df

    # Soft Gaussian window instead of hard cut (avoids ringing)
    window = np.exp(-0.5 * (dist / thickness_freq) ** 2)

    # Zero out DC — it's on every plane and carries no motion information
    V_filtered = V * window
    V_filtered[0, 0, 0] = 0

    # Reconstruct
    recon = np.real(np.fft.ifftn(V_filtered))

    # Energy map
    energy = recon ** 2
    if smooth > 0:
        energy = gaussian_filter(energy, sigma=[0, smooth, smooth])

    # Normalise per frame for display
    e_min, e_max = energy.min(), energy.max()
    energy_norm = (energy - e_min) / (e_max - e_min + 1e-12)

    threshold = np.percentile(energy_norm, 85)
    mask = energy_norm > threshold

    return recon, energy_norm, mask


# ── Motion compensated (unchanged but included for comparison) ────────────────

def motion_compensated_segmentation(video, vx, vy, diff_threshold=0.12, smooth=2.0):
    T, H, W = video.shape
    stabilised = np.zeros_like(video)
    for t in range(T):
        stabilised[t] = shift(video[t], shift=[-t * vy, -t * vx], mode="wrap")
    mean_frame = stabilised.mean(axis=0)
    residuals = np.abs(stabilised - mean_frame[np.newaxis])
    if smooth > 0:
        residuals = gaussian_filter(residuals, sigma=[0, smooth, smooth])
    mask = residuals < diff_threshold
    return stabilised, mean_frame, residuals, mask


# ── Demo ─────────────────────────────────────────────────────────────────────

np.random.seed(0)
W, H, T = 128, 128, 24
VX, VY = 2.5, 1.2     # pixels/frame — moderate velocity

objects = [
    {"cx": 20, "cy": 50, "vx": VX,  "vy": VY,  "r": 10, "intensity": 0.85},
    {"cx": 90, "cy": 80, "vx": VX,  "vy": VY,  "r":  8, "intensity": 0.80},
    {"cx": 65, "cy": 30, "vx": 0.0, "vy": 0.0, "r": 11, "intensity": 0.70},
]

video = make_video(W, H, T, objects, noise=0.02)
frame = T // 2

# ── Ground truth mask for comparison ──────────────────────────────────
yy, xx = np.mgrid[0:H, 0:W]
gt_mask = np.zeros((H, W), bool)
for obj in objects[:2]:           # only the moving balls
    cx = obj["cx"] + frame * obj["vx"]
    cy = obj["cy"] + frame * obj["vy"]
    gt_mask |= (xx - cx) ** 2 + (yy - cy) ** 2 <= obj["r"] ** 2

# ── Run methods ────────────────────────────────────────────────────────
print("Running discriminative phase segmentation...")
phase_score = phase_score_discriminative(video, VX, VY, smooth=2.0)
phase_mask  = phase_score[frame - 1] > np.percentile(phase_score[frame - 1], 85)

print("Running 3D Fourier plane segmentation...")
recon, energy, fourier_mask = fourier_plane_segmentation(
    video, VX, VY, plane_thickness=2.0, smooth=3.0)

print("Running motion-compensated segmentation...")
stabilised, mean_frame_img, residuals, mc_mask = motion_compensated_segmentation(
    video, VX, VY, diff_threshold=0.12, smooth=2.0)

# ── Plot ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(15, 11))
fig.suptitle(f"Motion Segmentation  |  vx={VX}, vy={VY} px/frame  |  frame {frame}",
                fontsize=13, fontweight="bold")

def show(ax, img, title, cmap="gray", vmin=0, vmax=1):
    ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax,
                origin="upper", interpolation="nearest")
    ax.set_title(title, fontsize=9)
    ax.axis("off")

def overlay(ax, frame_img, mask, title):
    rgb = np.stack([frame_img] * 3, axis=-1)
    rgb[mask, 0] = 1.0
    rgb[mask, 1] = 0.2
    rgb[mask, 2] = 0.2
    ax.imshow(rgb, origin="upper", interpolation="nearest")
    ax.set_title(title, fontsize=9); ax.axis("off")

# Row 0 – raw + ground truth
show(axes[0, 0], video[0],            "Frame 0")
show(axes[0, 1], video[frame],        f"Frame {frame}")
show(axes[0, 2], video[-1],           f"Frame {T-1}")
overlay(axes[0, 3], video[frame], gt_mask, "Ground truth (moving balls)")

# Row 1 – discriminative phase
vmin_s = phase_score.min(); vmax_s = phase_score.max()
show(axes[1, 0], phase_score[frame-1], "Phase disc. score",
        cmap="RdBu", vmin=vmin_s, vmax=vmax_s)
show(axes[1, 1], phase_score.mean(0),  "Phase score (mean t)",
        cmap="RdBu", vmin=vmin_s, vmax=vmax_s)
show(axes[1, 2], phase_mask.astype(float), "Phase mask (frame)")
overlay(axes[1, 3], video[frame], phase_mask, "Phase overlay")

# Row 2 – 3D Fourier
show(axes[2, 0], recon[frame],         "Fourier recon",  cmap="RdBu", vmin=-0.3, vmax=0.3)
show(axes[2, 1], energy[frame],        "Fourier energy", cmap="hot")
show(axes[2, 2], fourier_mask[frame].astype(float), "Fourier mask")
overlay(axes[2, 3], video[frame], fourier_mask[frame], "Fourier overlay")

plt.tight_layout()
plt.savefig(f"../results/{dataname}_motion_seg_fixed.png", dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved → {dataname}_motion_seg_fixed.png")